In [35]:

import json
from pathlib import Path
import pprint
import pandas as pd
from huggingface_hub import snapshot_download

In [22]:
DATASET_FOLDERS = (
    "full_persona",
    "question_catalog_and_human_response_csv",
    "wave_split",
    "raw_data",
    "LLM_simulation_results",
)

_DEMO_QIDS = {
    "QID11": "region",
    "QID12": "gender",
    "QID13": "age_band",
    "QID14": "education_level",
    "QID15": "race",
    "QID16": "us_citizen",
    "QID17": "marital_status",
    "QID18": "religion",
    "QID19": "religious_attendance",
    "QID20": "political_affiliation",
    "QID21": "income_band",
    "QID22": "political_views",
    "QID23": "household_size",
    "QID24": "employment_status",
}

In [5]:
# Use code from ingest.py
# Download full_persona folder
# Read the parquet to see the structure

In [27]:
def download_folder(folder: str, dest_dir: str | Path) -> Path:
    dest = Path(dest_dir).expanduser().resolve()
    dest.mkdir(parents=True, exist_ok=True)

    snapshot_download(
        repo_id="LLM-Digital-Twin/Twin-2K-500",
        repo_type="dataset",
        local_dir=dest,
        allow_patterns=[f"{folder}/**"],
    )

    return dest / folder

# Check full_persona folder content

In [ ]:
full_persona_dir = download_folder("full_persona", "~/Downloads/Twin-2K-500")

Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 45.65it/s]


In [18]:
chunks_dir = full_persona_dir / "chunks"

for parquet_path in sorted(chunks_dir.glob("*.parquet")):
    print(parquet_path)
    
    frame = pd.read_parquet(parquet_path)
    for record in frame.itertuples(index=False):
        raw_json = record.persona_json
        if raw_json is not None and not isinstance(raw_json, str):
            raw_json = json.dumps(raw_json)
        # print(raw_json)
        break
    break



/Users/goku/Downloads/Twin-2K-500/full_persona/chunks/persona_chunk_001.parquet


In [17]:
json.loads(raw_json)

[{'ElementType': 'Block',
  'BlockName': 'Demographics',
  'BlockType': 'Standard',
  'Questions': [{'QuestionID': 'QID11',
    'QuestionText': 'Which part of the United States do you currently live in?',
    'QuestionType': 'MC',
    'Options': ['Northeast (PA, NY, NJ, RI, CT, MA, VT, NH, ME)',
     'Midwest (ND, SD, NE, KS, MN, IA, MO, WI, IL, MI, IN, OH)',
     'South (TX, OK, AR, LA, KY, TN, MS, AL, WV, DC, MD, DE, VA, NC, SC, GA, FL)',
     'West (WA, OR, ID, MT, WY, CA, NV, UT, CO, AZ, NM)',
     'Pacific (HI, AK)'],
    'Settings': {'Selector': 'SAVR',
     'SubSelector': 'TX',
     'ForceResponse': 'ON'},
    'Answers': {'SelectedByPosition': 3,
     'SelectedText': 'South (TX, OK, AR, LA, KY, TN, MS, AL, WV, DC, MD, DE, VA, NC, SC, GA, FL)'}},
   {'QuestionID': 'QID12',
    'QuestionText': 'What is the sex that you were assigned at birth?',
    'QuestionType': 'MC',
    'Options': ['Male', 'Female'],
    'Settings': {'Selector': 'SAVR',
     'SubSelector': 'TX',
     'ForceRes

In [20]:
blocks = json.loads(raw_json)
demo_block = None
for block in blocks:
    if block.get("BlockName") == "Demographics":
        demo_block = block
        break

In [ ]:
by_qid = {q["QuestionID"]: q for q in demo_block.get("Questions", [])}
row: dict[str, object] = {}
for qid, column in _DEMO_QIDS.items():
    question = by_qid.get(qid)
    text = question.get("Answers").get("SelectedText")
    print(qid, text)

QID11 South (TX, OK, AR, LA, KY, TN, MS, AL, WV, DC, MD, DE, VA, NC, SC, GA, FL)
QID12 Male
QID13 18-29
QID14 Some college, no degree
QID15 White
QID16 Yes
QID17 Never been married
QID18 Protestant
QID19 Once or twice a month
QID20 Republican
QID21 $100,000 or more
QID22 Conservative
QID23 4
QID24 Student


# Check questions

In [29]:
catalog_dir = download_folder("question_catalog_and_human_response_csv", "~/Downloads/Twin-2K-500")
catalog_dir

Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 846.43it/s]


PosixPath('/Users/goku/Downloads/Twin-2K-500/question_catalog_and_human_response_csv')

In [30]:
def _read_response_csv_column_names(path: Path) -> set[str]:
    return set(pd.read_csv(path, nrows=0).columns) - {"pid"}


In [ ]:
catalog = json.loads((catalog_dir / "question_catalog.json").read_text())
wave4_column_names   = _read_response_csv_column_names(catalog_dir / "wave4_response.csv")


{'DB': 14, 'MC': 175, 'Matrix': 36, 'Slider': 3, 'TE': 28}


# Question Catalog Exploration

In [63]:
print(f"Number of questionIDs in the catalog: {len(catalog)}")

Number of questionIDs in the catalog: 256


In [89]:
# total questions
total_questions = sum(len(q["csv_columns"]) for q in catalog)
print(f"Total number of questions: {total_questions}")

Total number of questions: 760


In [ ]:
# unique question types
unique_question_types = set(question["QuestionType"] for question in catalog)
count_by_question_type = {question_type: sum(1 for q in catalog if q["QuestionType"] == question_type) for question_type in unique_question_types}
print(f"Number of unique question types: {len(unique_question_types)}")
print(f"Question types and their counts: {count_by_question_type}")

Number of unique question types: 5
Question types and their counts: {'MC': 175, 'DB': 14, 'Slider': 3, 'Matrix': 36, 'TE': 28}


## MCQs

In [83]:
MCQs = [q for q in catalog if q["QuestionType"] == "MC"]

mcq_blocks = [q["BlockName"] for q in MCQs]
mcq_blocks = set(mcq_blocks)    
mcq_blocks_count = {mcq_block: sum(1 for q in MCQs if q["BlockName"] == mcq_block) for mcq_block in mcq_blocks}
print(f"Number of unique MCQ blocks: {len(mcq_blocks)}")
pprint.pprint(mcq_blocks_count)

Number of unique MCQ blocks: 32
{'Absolute vs. relative - calculator': 1,
 'Absolute vs. relative - jacket': 1,
 'Allais Form 1': 1,
 'Allais Form 2': 1,
 'Anchoring - African countries high': 1,
 'Anchoring - African countries low ': 1,
 'Anchoring - redwood high': 1,
 'Anchoring - redwood low': 1,
 'Cognitive tests ': 50,
 'Demographics': 14,
 'Disease - gain ': 1,
 'Disease-loss': 1,
 'Economic preferences ': 18,
 'Less is More Gamble A ': 1,
 'Less is More Gamble B ': 1,
 'Less is More Gamble C ': 1,
 'Myside Ford': 1,
 'Myside German': 1,
 'Non-experimental heuristics and biases ': 2,
 'Outcome bias - failure': 1,
 'Outcome bias - success ': 1,
 'Personality ': 25,
 'Product Preferences - Pricing': 40,
 'Proportion dominance 1A ': 1,
 'Proportion dominance 1B': 1,
 'Proportion dominance 1C': 1,
 'Proportion dominance 2A': 1,
 'Proportion dominance 2B ': 1,
 'Proportion dominance 2C': 1,
 'WTA/WTP Thaler - WTP noncertainty': 1,
 'WTA/WTP Thaler problem - WTA certainty': 1,
 'WTA/WT

In [84]:
blocks_with_sub_questions = set([mcq["BlockName"] for mcq in MCQs if len(mcq["csv_columns"]) > 1])
print(f"Number of blocks with sub questions: {len(blocks_with_sub_questions)}")
pprint.pprint(blocks_with_sub_questions)

Number of blocks with sub questions: 2
{'Cognitive tests ', 'Personality '}


## DBQs

In [90]:
DBQs = [q for q in catalog if q["QuestionType"] == "DB"]

dbq_blocks = [q["BlockName"] for q in DBQs]
dbq_blocks = set(dbq_blocks)
dbq_blocks_count = {dbq_block: sum(1 for q in DBQs if q["BlockName"] == dbq_block) for dbq_block in dbq_blocks}
print(f"Number of unique DBQ blocks: {len(dbq_blocks)}")
pprint.pprint(dbq_blocks_count)

Number of unique DBQ blocks: 5
{'Cognitive tests ': 4,
 'Economic preferences ': 2,
 'Economic preferences - intro ': 2,
 'Personality ': 5,
 'Product Preferences - Pricing': 1}


In [92]:
print(f"Number of DBQs: {len(DBQs)}")

Number of DBQs: 14


In [94]:
dbqs_with_csv_columns = 0
for dbq in DBQs:
    if len(dbq["csv_columns"]) > 0:
        dbqs_with_csv_columns += 1
print(f"Number of DBQs with CSV columns: {dbqs_with_csv_columns}")

Number of DBQs with CSV columns: 0


In [95]:
for block in dbq_blocks:
    print(f"Block: {block}")
    for dbq in DBQs:
        if dbq["BlockName"] == block:
            pprint.pprint(dbq)
            print("--------------------------------")
            break

Block: Cognitive tests 
{'BlockName': 'Cognitive tests ',
 'QuestionID': 'QID241',
 'QuestionText': 'In this section, you will answer a series of 42 questions '
                 'that will test your performance on various cognitive tests.',
 'QuestionType': 'DB',
 'csv_columns': [],
 'is_descriptive': True,
 'source': 'wave1_3_persona_json'}
--------------------------------
Block: Economic preferences 
{'BlockName': 'Economic preferences ',
 'QuestionID': 'QID243',
 'QuestionText': 'The following 6 questions will concern amounts of money you '
                 'may receive in the future. In particular, these questions '
                 'will concern amounts of money that you may receive after '
                 'some delay, for example $5 received in 10 days’ time.',
 'QuestionType': 'DB',
 'csv_columns': [],
 'is_descriptive': True,
 'source': 'wave1_3_persona_json'}
--------------------------------
Block: Product Preferences - Pricing
{'BlockName': 'Product Preferences - Pricing',
 

## Sliders

In [96]:
sliders = [q for q in catalog if q["QuestionType"] == "Slider"]
print(f"Number of sliders: {len(sliders)}")

Number of sliders: 3


In [99]:
slider_blocks = [q["BlockName"] for q in sliders]
slider_blocks = set(slider_blocks)
slider_blocks_count = {slider_block: sum(1 for q in sliders if q["BlockName"] == slider_block) for slider_block in slider_blocks}
print(f"Number of unique slider blocks: {len(slider_blocks)}")
pprint.pprint(slider_blocks_count)

Number of unique slider blocks: 3
{'Base-rate 30 engineers ': 1,
 'Base-rate 70 engineers': 1,
 'Non-experimental heuristics and biases ': 1}


In [100]:
pprint.pprint(sliders)

[{'BlockName': 'Base-rate 30 engineers ',
  'QuestionID': 'QID154',
  'QuestionText': 'A panel of psychologist have interviewed and administered '
                  'personality tests to 30 engineers and 70 lawyers, all '
                  'successful in their respective fields. On the basis of this '
                  'information, thumbnail descriptions of the 30 engineers and '
                  '70 lawyers have been written. Below is one description, '
                  'chosen at random from the 100 available descriptions. Jack '
                  'is a 45-year-old man. He is married and has four children. '
                  'He is generally conservative, careful, and ambitious. He '
                  'shows no interest in political and social issues and spends '
                  'most of his free time on his many hobbies which include '
                  'home carpentry, sailing, and mathematical puzzles. The '
                  'probability that Jack is one of the 30 engineers

## Matrix

In [101]:
matrix_questions = [q for q in catalog if q["QuestionType"] == "Matrix"]
print(f"Number of matrix questions: {len(matrix_questions)}")

Number of matrix questions: 36


In [102]:
matrix_blocks = [q["BlockName"] for q in matrix_questions]
matrix_blocks = set(matrix_blocks)
matrix_blocks_count = {matrix_block: sum(1 for q in matrix_questions if q["BlockName"] == matrix_block) for matrix_block in matrix_blocks}
print(f"Number of unique matrix blocks: {len(matrix_blocks)}")
pprint.pprint(matrix_blocks_count)

Number of unique matrix blocks: 8
{'Economic preferences ': 13,
 'False consensus ': 1,
 'Linda -no conjunction ': 1,
 'Linda-conjunction': 1,
 'Non-experimental heuristics and biases ': 2,
 'Personality ': 16,
 'Probability matching vs. maximizing - Problem 1': 1,
 'Probability matching vs. maximizing - Problem 2': 1}


In [103]:
for block in matrix_blocks:
    print(f"Block: {block}")
    for matrix_question in matrix_questions:
        if matrix_question["BlockName"] == block:
            pprint.pprint(matrix_question)
            print("--------------------------------")
            break

Block: Linda -no conjunction 
{'BlockName': 'Linda -no conjunction ',
 'Columns': ['Extremely improbable',
             'Very improbable',
             'Somewhat probable',
             'Moderately probable',
             'Very probable',
             'Extremely probable'],
 'QuestionID': 'QID159',
 'QuestionText': 'Linda is 31 years old, single, outspoken, and very bright. '
                 'She majored in philosophy. As a student, she was deeply '
                 'concerned with issues of discrimination and social justice, '
                 'and also participated in anti-nuclear demonstrations. Please '
                 'complete the statements below.',
 'QuestionType': 'Matrix',
 'Rows': ['It is ___ that Linda is a teacher in an elementary school',
          'It is ___ that Linda works in a bookstore and takes Yoga classes',
          'It is ___ that Linda is a bank teller'],
 'Settings': {'ForceResponse': 'ON',
              'Selector': 'Likert',
              'SubSelector': 'Si

## TE

In [105]:
TES = [q for q in catalog if q["QuestionType"] == "TE"]
print(f"Number of TEs: {len(TES)}")

Number of TEs: 28


In [106]:
te_blocks = [q["BlockName"] for q in TES]
te_blocks = set(te_blocks)
te_blocks_count = {te_block: sum(1 for q in TES if q["BlockName"] == te_block) for te_block in te_blocks}
print(f"Number of unique TE blocks: {len(te_blocks)}")
pprint.pprint(te_blocks_count)

Number of unique TE blocks: 10
{'Anchoring - African countries high': 1,
 'Anchoring - African countries low ': 1,
 'Anchoring - redwood high': 1,
 'Anchoring - redwood low': 1,
 'Cognitive tests ': 15,
 'Economic preferences ': 3,
 'Forward Flow': 1,
 'Personality ': 3,
 'Sunk cost - no': 1,
 'Sunk cost - yes': 1}


In [108]:
# pprint.pprint(TES)
for block in te_blocks:
    print(f"Block: {block}")
    for te in TES:
        if te["BlockName"] == block:
            pprint.pprint(te)
            print("--------------------------------")
            break

Block: Cognitive tests 
{'BlockName': 'Cognitive tests ',
 'QuestionID': 'QID52',
 'QuestionText': 'Emily’s father has three daughters. The first two are named '
                 'April and May. What is the third daughter’s name?',
 'QuestionType': 'TE',
 'Settings': {'ContentType': '', 'ForceResponse': 'ON', 'Selector': 'SL'},
 'csv_columns': ['QID52_TEXT'],
 'source': 'wave1_3_persona_json'}
--------------------------------
Block: Economic preferences 
{'BlockName': 'Economic preferences ',
 'QuestionID': 'QID271',
 'QuestionText': 'We are now interested in what you were thinking about while '
                 'deciding how much of the money to keep for yourself and how '
                 'much to send to another (anonymous) person, and when '
                 'deciding how much would you keep for yourself and how much '
                 'would send back to the other person. Any thought is fine; '
                 'simply list what it was that you were thinking about while '
        